In [ ]:
import sys
sys.path.append("..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
from src.utils import qlike

df = pd.read_csv("../data/GSPC_processed.csv", index_col=0, parse_dates=True)
r = df["log_return"]

split = int(len(r) * 0.8)          # chronological 80/20 split
test_idx = r.index[split:]
print("Train:", r.index[0].date(), "->", r.index[split - 1].date())
print("Test: ", test_idx[0].date(), "->", test_idx[-1].date())

In [ ]:
specs = {
    "GARCH(1,1)": dict(vol="GARCH", p=1, q=1),
    "EGARCH":     dict(vol="EGARCH", p=1, o=1, q=1),
    "GJR-GARCH":  dict(vol="GARCH", p=1, o=1, q=1),
}

results = {}
for name, spec in specs.items():
    am = arch_model(r, mean="Constant", dist="t", **spec)
    results[name] = am.fit(last_obs=r.index[split - 1], disp="off")

comparison = pd.DataFrame({
    n: {"LogLik": res.loglikelihood, "AIC": res.aic, "BIC": res.bic}
    for n, res in results.items()
}).T
comparison

In [ ]:
print(results["GARCH(1,1)"].summary())
print(results["GJR-GARCH"].params.round(4))
print(results["EGARCH"].params.round(4))

In [ ]:
forecasts = {}
for name, res in results.items():
    f = res.forecast(horizon=1, start=r.index[split - 1], reindex=False)
    # a forecast made on day t is for day t+1, so shift by one
    forecasts[name] = f.variance["h.1"].shift(1).loc[test_idx]

# Baseline: yesterday's 21-day rolling variance
forecasts["Rolling 21d var"] = r.rolling(21).var().shift(1).loc[test_idx]

fc = pd.DataFrame(forecasts).dropna()
fc.head()

In [ ]:
proxy = (r.loc[fc.index]) ** 2

rows = {}
for name in fc.columns:
    err = fc[name] - proxy
    rows[name] = {
        "RMSE": np.sqrt((err ** 2).mean()),
        "MAE": err.abs().mean(),
        "QLIKE": qlike(proxy, fc[name]),
    }
metrics = pd.DataFrame(rows).T.sort_values("QLIKE")
metrics.round(4)

In [ ]:
realized = df["realized_vol_21d"].loc[fc.index]

fig, ax = plt.subplots(figsize=(12, 5))
realized.plot(ax=ax, color="black", lw=1.5, label="21d realized vol")
for name in ["GARCH(1,1)", "EGARCH", "GJR-GARCH"]:
    (np.sqrt(fc[name] * 252)).plot(ax=ax, lw=0.9, alpha=0.8, label=name)
ax.set_title("Test period: model-forecast vs realized volatility (annualized, %)")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
fc.to_csv("../data/garch_forecasts.csv")